# 6-bob. 6-BOB. FISHING VA SPAMNI SUN'IY INTELLEKT YORDAMIDA ANIQLASH
## Amaliy misol: o'zbek tilidagi xabarlarni tasniflash

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/06-bob/6-1_ozbek-tilidagi-xabarlarni-tasniflash.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 6-bob.*

Quyidagi dasturda o'zbek tilidagi qisqa xabarlarni fishing va oddiy xabarlarga ajratuvchi model quriladi. Matnlar TF-IDF usuli yordamida vektorga aylantiriladi, tasniflash uchun esa logistik regressiyadan foydalaniladi. Logistik regressiya tanlanishiga sabab shuki, uning qarorini tushuntirish oson: model har bir so'zga vazn beradi va biz qaysi so'zlar fishingga xos ekanini bevosita ko'ra olamiz. Dastur amaliy mashg'ulotning A3 va A6 mavzulariga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
# 1. O'quv ma'lumotlari: 1 - fishing, 0 - oddiy xabar
xabarlar = [
    ("Hurmatli mijoz, kartangiz bloklandi. Tiklash uchun havolaga kiring", 1),
    ("Tabriklaymiz! Siz yutuq yutdingiz, olish uchun karta raqamini yuboring", 1),
    ("Hisobingiz shubhali faollik sababli to'xtatildi, zudlik bilan tasdiqlang", 1),
    ("Soliq qarzingiz bor, jarima to'lamaslik uchun 24 soat ichida havolaga kiring", 1),
    ("Davlat yordam puli tayyor, olish uchun SMS kodni yuboring", 1),
    ("Posilkangiz omborda qoldi, yetkazish uchun to'lovni havola orqali amalga oshiring", 1),
    ("Ertaga soat 10 da kafedra yig'ilishi bo'ladi", 0),
    ("Amaliy mashg'ulot topshiriqlarini juma kuniga qadar yuboring", 0),
    ("Kutubxonadan olgan kitobingizni qaytarish muddati keldi", 0),
    ("Dars jadvali o'zgardi, yangi jadvalni sayt orqali ko'ring", 0),
    ("Konferensiya tezislarini qabul qilish muddati uzaytirildi", 0),
    ("Onang kechqurun uyga erta kelishingni so'radi", 0),
]
matnlar = [m for m, _ in xabarlar]
y = np.array([b for _, b in xabarlar])

In [3]:
# 2. TF-IDF vektorlashtirish (so'zlar va ikki so'zli iboralar)
# O'zbek so'zlaridagi tutuq belgisini (') so'z ichida saqlab qolamiz
vekt = TfidfVectorizer(ngram_range=(1, 2), lowercase=True, token_pattern=r"(?u)\b\w[\w']*\b")
X = vekt.fit_transform(matnlar)

In [4]:
# 3. Logistik regressiya modelini o'qitish
model = LogisticRegression(C=10, max_iter=1000).fit(X, y)

In [5]:
# 4. Model qaysi so'zlarni fishing belgisi deb o'rgandi?
sozlar = np.array(vekt.get_feature_names_out())
tartib = np.argsort(model.coef_[0])
print("Fishingga eng xos so'zlar:", ", ".join(sozlar[tartib[-8:]][::-1]))
print("Oddiy xabarga eng xos so'zlar:", ", ".join(sozlar[tartib[:6]]))

Fishingga eng xos so'zlar: uchun, olish, olish uchun, havolaga kiring, kiring, havolaga, shubhali, sababli to'xtatildi
Oddiy xabarga eng xos so'zlar: muddati, erta, kelishingni, erta kelishingni, kechqurun, kelishingni so'radi


In [6]:
# 5. Yangi xabarlarni tekshirish
yangi = [
    "Kartangiz vaqtincha bloklandi, tasdiqlash uchun kodni yuboring",
    "Juma kuni kafedra yig'ilishiga kelishni unutmang",
]
for m, e in zip(yangi, model.predict_proba(vekt.transform(yangi))[:, 1]):
    print(f"{e:.2f}  {m}")

0.73  Kartangiz vaqtincha bloklandi, tasdiqlash uchun kodni yuboring
0.32  Juma kuni kafedra yig'ilishiga kelishni unutmang


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Fishingga eng xos so'zlar: uchun, olish, olish uchun, havolaga kiring, kiring, havolaga, shubhali, sababli to'xtatildi
Oddiy xabarga eng xos so'zlar: muddati, erta, kelishingni, erta kelishingni, kechqurun, kelishingni so'radi
0.73  Kartangiz vaqtincha bloklandi, tasdiqlash uchun kodni yuboring
0.32  Juma kuni kafedra yig'ilishiga kelishni unutmang
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 6-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.